<a href="https://colab.research.google.com/github/esprydi/sentimen-analisis-shopee-playstore/blob/master/Sentiment_Analisis_shopee_Playstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-play-scraper

# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews, Sort, reviews_all

import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
import numpy as np  # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score

import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

#Scraping Dataset

In [2]:
#mengimpor pustaka google_play_scrapper

from google_play_scraper import app, reviews_all, Sort

# Mengambil semua ulasan dari aplikasi
scrapreview = reviews(
    'com.shopee.id',
    lang='id',
    country='id',
    sort=Sort.MOST_RELEVANT,
    count=20000
)

In [3]:
# Menyimpan ulasan dalam file CSV
import csv

with open('ulasan_shopee_app.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['review'])
    for review in scrapreview[0]: # Access the first element of the tuple which contains the list of reviews
        writer.writerow([review['content']])

# Loading Dataset

In [4]:
app_reviews_df = pd.DataFrame(scrapreview[0])
app_reviews_df.shape
app_reviews_df.head()
app_reviews_df.to_csv('shopee_app_reviews.csv', index=None, header=True)

# Membuat DataFrame dari hasil scrapreview
app_reviews_df = pd.DataFrame(scrapreview[0])

# Menghitung jumlah baris dan kolom dalam DataFrame
jumlah_ulasan, jumlah_kolom = app_reviews_df.shape

# Menampilkan hasil
print(f"Jumlah ulasan: {jumlah_ulasan}")
print

Jumlah ulasan: 20000


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [5]:
# Menampilkan lima baris pertama
app_reviews_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,0f32ac46-b94e-42a5-83ad-e795d272665c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Bagus,Asli keamanannya bagus, aman juga lancar...",5,1,3.68.42,2026-03-10 06:30:09,"Hi kak Monika Irma, makasih bintang 5 nya, ak...",2026-03-10 08:45:50,3.68.42
1,675d7447-9df2-4eba-85b4-24660d842bb5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Asli keamanannya bagus, aman juga lancar semua...",5,2,3.68.42,2026-03-03 12:39:10,"Hi kak misterius_, makasih ya buat review bint...",2026-03-03 14:25:07,3.68.42
2,faa9dab8-1efb-4209-80b8-2cbdbcd152e9,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Banyak error, ya ampun lemot banget terutama b...",2,0,3.68.42,2026-03-09 11:56:59,"Hai kak Terrysha Cindy Auxilia, makasih ya bua...",2026-03-09 12:48:10,3.68.42
3,e5fa64c2-fcee-429e-8775-0644f584a59b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Top,Saya sangat suka belanja disini selain ban...",5,1,3.68.42,2026-03-10 07:31:47,"Hi kak Pipi Papi, maaf ya sudah buat kmu gak n...",2026-03-10 08:21:09,3.68.42
4,de9f9f78-fa9b-4521-b460-662470082749,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"shopee bagus ada fitur cod cek dulu ,minim pen...",4,0,3.69.32,2026-03-09 21:43:49,"Hai Kak Ika Pitriyani, maaf atas ketidaknyaman...",2026-03-10 02:39:56,3.69.32


In [6]:
# Menampilkan informasi
app_reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              20000 non-null  object        
 1   userName              20000 non-null  object        
 2   userImage             20000 non-null  object        
 3   content               20000 non-null  object        
 4   score                 20000 non-null  int64         
 5   thumbsUpCount         20000 non-null  int64         
 6   reviewCreatedVersion  19877 non-null  object        
 7   at                    20000 non-null  datetime64[ns]
 8   replyContent          18741 non-null  object        
 9   repliedAt             18741 non-null  datetime64[ns]
 10  appVersion            19877 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 1.7+ MB


In [7]:
# Membuat DataFrame baru dengan menghapus baris yang memiliki nilai yang hilang

clean_df = app_reviews_df.dropna()

In [8]:
# Menghapus baris duplikat
clean_df = clean_df.drop_duplicates()

# Menghitung jumlah baris dan kolom dalam DataFrame clean_df setelah menghapus duplikat
jumlah_ulasan,jumlah_kolom = clean_df.shape

clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18624 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              18624 non-null  object        
 1   userName              18624 non-null  object        
 2   userImage             18624 non-null  object        
 3   content               18624 non-null  object        
 4   score                 18624 non-null  int64         
 5   thumbsUpCount         18624 non-null  int64         
 6   reviewCreatedVersion  18624 non-null  object        
 7   at                    18624 non-null  datetime64[ns]
 8   replyContent          18624 non-null  object        
 9   repliedAt             18624 non-null  datetime64[ns]
 10  appVersion            18624 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 1.7+ MB


# Preprocessing Text

In [9]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# --- 1. INISIALISASI ---
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def filteringText(text_tokens): # Menghapus stopwords dalam teks
    # Ambil stopwords dasar
    listStopwords = set(stopwords.words('indonesian'))
    # --- PENYELAMATAN NEGASI ---
    # Daftar kata yang TIDAK BOLEH dihapus karena penting untuk makna negatif
    negasi_penting = {"tidak", "bukan", "kurang", "jangan", "ga", "gak", "nggak"}
    for kata in negasi_penting:
        if kata in listStopwords:
            listStopwords.remove(kata)
    # Tambahkan kata-kata sampah spesifik Shopee/Playstore
    listStopwords.update(['dan', 'saya', 'dll', 'dkk', 'dsb', 'dst', 'yg', 'itu', 'ini',
        'adalah', 'ya', 'kak', 'shopee', 'aplikasi', 'nya', 'sih'])    # Filter: Simpan kata jika TIDAK ada di listStopwords ATAU mengandung "_"
    filtered = [w for w in text_tokens if (w not in listStopwords) or ("_" in w)]

    return filtered # Mengembalikan list, bukan string

# Buat kamus kosong untuk menyimpan kata yang sudah di-stem
stem_cache = {}

def stemmingText(text):
    words = text.split()
    stemmed_words = []

    for word in words:
        if word not in stem_cache:
            # Jika kata belum pernah di-stem, hitung dan simpan ke cache
            stem_cache[word] = stemmer.stem(word)
        stemmed_words.append(stem_cache[word])

    return ' '.join(stemmed_words)

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

def reduce_lengthening(text): # Normalisasi karakter berulang misal : bagusss jadi bagus
    pattern = re.compile(r"(.)\1{2,}")
    return pattern.sub(r"\1\1", text)

def handle_negation(text):
    # Gunakan hanya kata baku karena slang sudah diperbaiki di step sebelumnya
    # "ga", "gak", "nggak" TIDAK perlu ada di sini jika sudah diubah jadi "tidak"
    negation_words = ["tidak", "bukan", "kurang", "jangan"]

    words = text.split()
    new_words = []
    i = 0
    while i < len(words):
        if words[i] in negation_words and i + 1 < len(words):
            combined_word = words[i] + "_" + words[i + 1]
            new_words.append(combined_word)
            i += 2
        else:
            new_words.append(words[i])
            i += 1
    return ' '.join(new_words)



In [10]:
slangwords = {"ga": "tidak", "gak": "tidak", "gakk": "tidak", "nggak": "tidak", "tdk": "tidak",
    "bgt": "banget", "bangettt": "banget", "skali": "sekali",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "kalo": "kalau", "kl": "kalau", "klo": "kalau",
    "jd": "jadi", "jdi": "jadi",
    "kmrn": "kemarin", "skrg": "sekarang", "skr": "sekarang",
    "lemot": "lambat", "lola": "lambat",
    "ongkir": "ongkos kirim", "cod": "bayar di tempat",
    "bales": "balas", "admin": "petugas",
    "pas": "ketika", "tp": "tapi", "tapi": "tetapi"}
def fix_slangwords(text):
    words = text.split()
    fixed_words = [slangwords[word] if word in slangwords else word for word in words]
    return ' '.join(fixed_words)


In [11]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

# 1. Membersihkan teks (Hapus simbol, URL, Angka)
clean_df['step1_clean'] = clean_df['content'].apply(cleaningText)

# 2. Case Folding (Kecilkan huruf)
clean_df['step2_fold'] = clean_df['step1_clean'].apply(casefoldingText)

# 3. Reduce Lengthening (Bagusss -> Baguss)
clean_df['step3_reduce'] = clean_df['step2_fold'].apply(reduce_lengthening)

# 4. Fix Slangwords (ga -> tidak)
# Note: Fungsi ini masih pakai .split() internal untuk memproses kata
clean_df['step4_slang'] = clean_df['step3_reduce'].apply(fix_slangwords)

# 5. Handle Negation (tidak bagus -> tidak_bagus)
clean_df['step5_negation'] = clean_df['step4_slang'].apply(handle_negation)

# 6. TOKENIZING (DI SINI TEMPATNYA)
# Memecah kalimat menjadi list kata yang bersih dan sudah terikat negasi
clean_df['step6_token'] = clean_df['step5_negation'].apply(tokenizingText)

# 7. Filtering (Hapus Stopwords dari LIST token)

clean_df['step7_filter'] = clean_df['step6_token'].apply(filteringText)
# 8. To Sentence (Kembalikan ke kalimat untuk dihitung skor leksikonnya)
clean_df['step8_sentence'] = clean_df['step7_filter'].apply(toSentence)

# 9. Stemming (Opsional: Mengecilkan kata ke bentuk dasar)
# Catatan: Jika data 18rb, proses ini mungkin butuh waktu 1-2 menit
clean_df['text_akhir'] = clean_df['step8_sentence'].apply(stemmingText)

display(clean_df[['content', 'text_akhir']].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


KeyboardInterrupt: 

# Pelabelan

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# 1. SETUP MODEL & TOKENIZER
pretrained = "mdhugol/indonesia-bert-sentiment-classification"
model = AutoModelForSequenceClassification.from_pretrained(pretrained)
tokenizer = AutoTokenizer.from_pretrained(pretrained)

# Pastikan device=0 jika menggunakan GPU di Colab (Sangat disarankan untuk 10rb data)
device = 0 if torch.cuda.is_available() else -1
sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

# Index label sesuai dokumentasi
label_index = {'LABEL_0': 'Positive', 'LABEL_1': 'Neutral', 'LABEL_2': 'Negative'}

from torch.utils.data import DataLoader

# --- 2. FUNGSI PROSES BATCH (SENTIMEN + EMBEDDING) ---
def get_sentiment_and_embeddings(texts, batch_size=32):
    model.eval()
    model.to(device)

    all_labels = []
    all_scores = []
    all_embeddings = []

    # Memproses data dalam batch agar RAM tidak meledak
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]

        # Tokenisasi
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

        with torch.no_grad():
            # Kita minta model mengeluarkan 'hidden_states' untuk embedding
            outputs = model(**inputs, output_hidden_states=True)

        # 1. Ekstrak Sentimen (Logits)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        scores, predictions = torch.max(probs, dim=-1)

        # 2. Ekstrak Embedding (Mengambil rata-rata hidden state terakhir / Mean Pooling)
        # last_hidden_state memiliki shape: [batch_size, sequence_length, 768]
        last_hidden = outputs.hidden_states[-1]
        mean_embedding = torch.mean(last_hidden, dim=1)

        all_labels.extend([label_index[f'LABEL_{p.item()}'] for p in predictions])
        all_scores.extend(scores.cpu().numpy())
        all_embeddings.extend(mean_embedding.cpu().numpy())

    return all_labels, all_scores, all_embeddings

# Jalankan Proses
list_texts = clean_df['content'].fillna("").astype(str).tolist()
labels, scores, embeddings = get_sentiment_and_embeddings(list_texts)

# --- 3. SIMPAN KE DATAFRAME ---
clean_df['sentiment'] = labels
clean_df['confidence_score'] = scores
clean_df['embedding'] = embeddings # Ini akan berisi list angka (vektor 768 dimensi)

print("\nSelesai! Sentimen dan Embedding berhasil diekstrak.")

# --- 4. CEK HASIL ---
print("\nDistribusi Sentimen (3 Kategori):")
print(clean_df['sentiment'].value_counts())
display(clean_df[['content', 'score', 'sentiment']].head())

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ambil hanya ulasan yang negatif
negative_text = " ".join(clean_df[clean_df['sentiment'] == 'Negative']['content'])

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(negative_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Kata-kata yang Paling Sering Muncul di Ulasan Negatif")
plt.show()

In [ ]:
# Menghitung jumlah ulasan untuk setiap kategori sentimen
sentiment_counts = clean_df['sentiment'].value_counts()
display(sentiment_counts)

# Memvisualisasikan distribusi sentimen
plt.figure(figsize=(8, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette='viridis')
plt.title('Distribution of Sentiment Labels')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()

#Eksplorasi Label

# Data Splitting dan Ekstraksi Fitur dengN TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF butuh teks dalam bentuk string tunggal
text_samples = clean_df['text_akhir'].apply(lambda x: ' '.join(x))

# Kita gunakan 300 fitur agar "apple-to-apple" dengan Word2Vec
tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,      # Naikkan drastis dari 300
    ngram_range=(1, 2),     # Membaca "bagus" DAN "tidak bagus"
    min_df=2                # Abaikan kata yang muncul kurang dari 2 kali
)
X_tfidf = tfidf_vectorizer.fit_transform(text_samples).toarray()

# Get the IndoBERT confidence scores and reshape them
indobert_score = clean_df['confidence_score'].values.reshape(-1, 1)

# Gabungkan dengan skor IndoBERT
X_combined_tfidf = np.hstack((X_tfidf, indobert_score))

print("Skenario 2 Siap: TF-IDF (300) + IndoBERT Score (1) = 301 Fitur")

#SMOTE TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

# Define the target variable 'y'
y = clean_df['sentiment']

# 1. Bagi data menjadi data Latih dan data Uji (WAJIB sebelum SMOTE)
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_combined_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Distribusi awal (Train TF-IDF): {Counter(y_train_tfidf)}")

In [ ]:
from imblearn.over_sampling import SMOTE

# 2. Terapkan SMOTE pada data Latih (X_train) saja
# SMOTE akan membuat data Neutral "buatan" agar jumlahnya sama dengan Positive
smote_tfidf = SMOTE(random_state=42)
X_train_res_tfidf, y_train_res_tfidf = smote_tfidf.fit_resample(X_train_tfidf, y_train_tfidf)

print(f"Distribusi setelah SMOTE (Train TF-IDF): {Counter(y_train_res_tfidf)}")

### Membuat Fitur Word2Vec

Langkah ini akan menghasilkan representasi vektor untuk setiap ulasan menggunakan model Word2Vec yang dilatih dari data teks yang sudah dibersihkan.

In [ ]:
!pip install gensim
from gensim.models import Word2Vec

# 1. Siapkan data untuk Word2Vec
# Kita akan menggunakan kolom 'step7_filter' yang berisi list kata-kata tokenized
word2vec_data = clean_df['text_akhir'].tolist()

# 2. Latih model Word2Vec
# vector_size: dimensi vektor (sesuai komentar 300 fitur)
# window: jarak maksimum antara kata target dan kata konteks
# min_count: mengabaikan semua kata dengan frekuensi total kurang dari ini
# workers: menggunakan jumlah core CPU untuk pelatihan (bisa disesuaikan)
print("Melatih model Word2Vec...")
model_w2v = Word2Vec(sentences=word2vec_data, vector_size=300, window=5, min_count=5, workers=4, seed=42)
print("Model Word2Vec berhasil dilatih.")

# 3. Fungsi untuk mendapatkan rata-rata vektor Word2Vec untuk sebuah dokumen (ulasan)
def document_vector(word_list, model):
    vectors = []
    for word in word_list:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size) # Mengembalikan vektor nol jika tidak ada kata yang ditemukan

# 4. Terapkan fungsi untuk setiap ulasan untuk membuat fitur X_w2v
print("Membuat vektor dokumen (X_w2v)...")
X_w2v = np.array([document_vector(words, model_w2v) for words in word2vec_data])
print(f"X_w2v berhasil dibuat dengan bentuk: {X_w2v.shape}")

# Verifikasi bahwa X_w2v memiliki 300 fitur
print(f"Jumlah fitur Word2Vec (kolom X_w2v): {X_w2v.shape[1]}")

# Tampilkan 5 baris pertama dari X_w2v (opsional, untuk inspeksi)
# display(pd.DataFrame(X_w2v).head())


In [ ]:
import numpy as np

# A. Ambil fitur dari Word2Vec yang sudah kamu buat sebelumnya (300 kolom)
# Pastikan X_w2v sudah di-run kodenya
X_w2v_feature = X_w2v

# B. Ambil skor keyakinan IndoBERT sebagai fitur tambahan (1 kolom)
# Kita gunakan .reshape(-1, 1) agar bentuknya jadi satu kolom tegak
indobert_score_feature = clean_df['confidence_score'].values.reshape(-1, 1)

# C. GABUNGKAN (Total 301 Fitur)
X_combined_word2vec = np.hstack((X_w2v_feature, indobert_score_feature))

# D. Siapkan Target (Label yang mau ditebak)
y = clean_df['sentiment']

print("--- KONFIGURASI FINAL ---")
print(f"Total Baris Data: {X_combined_word2vec.shape[0]}")
print(f"Total Fitur (X): {X_combined_word2vec.shape[1]} (300 W2V + 1 IndoBERT Score)")
print(f"Target (y): Menggunakan label dari IndoBERT")

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter


# 1. Bagi data menjadi data Latih dan data Uji (WAJIB sebelum SMOTE)
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_combined_word2vec, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Distribusi awal (Train Word2Vec): {Counter(y_train_w2v)}")

In [ ]:
from imblearn.over_sampling import SMOTE

# 2. Terapkan SMOTE pada data Latih (X_train) saja
# SMOTE akan membuat data Neutral "buatan" agar jumlahnya sama dengan Positive
smote = SMOTE(random_state=42)
X_train_res_word2vec, y_train_res_word2vec = smote.fit_resample(X_train_w2v, y_train_w2v)

print(f"Distribusi setelah SMOTE (Train Word2Vec): {Counter(y_train_res_word2vec)}")

# Modeling Menggunakan TF IDF

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded_tfidf = le.fit_transform(y_train_res_tfidf)
y_test_encoded_tfidf = le.transform(y_test_tfidf)

# Catatan:
# 0 biasanya Negative, 1 Neutral, 2 Positive (tergantung urutan abjad)
print("Urutan kelas:", le.classes_)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score # Added accuracy_score

# 1. Inisialisasi Model
xgb_model_tfidf = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.02,     # Belajar sangat detail
    max_depth=12,           # Pohon lebih dalam untuk menangkap pola Neutral
    gamma=0.2,                # Menambah regularisasi agar tidak overfit
    subsample=0.8,          # Mengambil 80% data secara acak agar tidak overfit
    colsample_bytree=0.7,   # Mengambil 80% fitur secara acak
    objective='multi:softprob', # Changed to binary:logistic for 2 classes
    num_class=3,
    random_state=42,
    n_jobs=-1
)

# 2. Training (Pakai data SMOTE yang sudah di-encode)
xgb_model_tfidf.fit(X_train_res_tfidf, y_train_encoded_tfidf)

# 3. Prediksi
y_pred_xgb_tfidf = xgb_model_tfidf.predict(X_test_tfidf) # Changed to X_test_tfidf

# 4. Evaluasi (Ubah kembali angka ke teks label asli)
print('XGBoost Accuracy:', accuracy_score(y_test_encoded_tfidf, y_pred_xgb_tfidf))
print(classification_report(y_test_encoded_tfidf, y_pred_xgb_tfidf, target_names=le.classes_))

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Inisialisasi Model
lgbm_model_tfidf = LGBMClassifier(
    n_estimators=1200,       # Tambah pohon agar belajar lebih lama
    learning_rate=0.03,      # Belajar lebih teliti
    num_leaves=150,          # Leaf-wise growth yang lebih kompleks (khas LightGBM)
    max_depth=12,            # Batasi kedalaman agar tidak overfit
    min_data_in_leaf=20,     # Mencegah pohon terlalu spesifik pada satu data
    feature_fraction=0.8,    # Random sampling fitur (mirip colsample di XGBoost)
    force_col_wise=True,
    random_state=42,
    n_jobs=-1,
    # class_weight='balanced' #DIHAPUS karena kita pakai data SMOTE
)

# 2. Training (WAJIB PAKAI HASIL SMOTE)
# Gunakan X_train_res dan y_train_encoded_res
lgbm_model_tfidf.fit(X_train_res_tfidf, y_train_encoded_tfidf)

# 3. Prediksi
y_pred_lgbm_tfidf = lgbm_model_tfidf.predict(X_test_tfidf)

# 4. Evaluasi
print('LightGBM Accuracy:', accuracy_score(y_test_encoded_tfidf, y_pred_lgbm_tfidf))
print("\nLaporan Klasifikasi LightGBM:")
print(classification_report(y_test_encoded_tfidf, y_pred_lgbm_tfidf, target_names=le.classes_))

# Modeling Menggunakan Word2Vec

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded_word2vec = le.fit_transform(y_train_res_word2vec)
y_test_encoded_word2vec = le.transform(y_test_w2v)

# Catatan:
# 0 biasanya Negative, 1 Neutral, 2 Positive (tergantung urutan abjad)
print("Urutan kelas:", le.classes_)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score # Added accuracy_score

# 1. Inisialisasi Model
xgb_model_word2vec = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.02,     # Belajar sangat detail
    max_depth=12,           # Pohon lebih dalam untuk menangkap pola Neutral
    gamma=0.2,                # Menambah regularisasi agar tidak overfit
    subsample=0.8,          # Mengambil 80% data secara acak agar tidak overfit
    colsample_bytree=0.8,   # Mengambil 80% fitur secara acak
    objective='multi:softprob', # Changed to binary:logistic for 2 classes
    num_class=3,
    random_state=42,
    n_jobs=-1
)

# 2. Training (Pakai data SMOTE yang sudah di-encode)
xgb_model_word2vec.fit(X_train_res_word2vec, y_train_encoded_word2vec)

# 3. Prediksi
y_pred_xgb_word2vec = xgb_model_word2vec.predict(X_test_w2v) # Changed to X_test_w2v

# 4. Evaluasi (Ubah kembali angka ke teks label asli)
print('XGBoost Accuracy:', accuracy_score(y_test_encoded_word2vec, y_pred_xgb_word2vec))
print(classification_report(y_test_encoded_word2vec, y_pred_xgb_word2vec, target_names=le.classes_))

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Inisialisasi Model
lgbm_model_word2vec = LGBMClassifier(
    n_estimators=1200,       # Tambah pohon agar belajar lebih lama
    learning_rate=0.03,      # Belajar lebih teliti
    num_leaves=150,          # Leaf-wise growth yang lebih kompleks (khas LightGBM)
    max_depth=12,            # Batasi kedalaman agar tidak overfit
    min_data_in_leaf=20,     # Mencegah pohon terlalu spesifik pada satu data
    feature_fraction=0.8,    # Random sampling fitur (mirip colsample di XGBoost)
    force_col_wise=True,
    random_state=42,
    n_jobs=-1,
    # class_weight='balanced' #DIHAPUS karena kita pakai data SMOTE
)

# 2. Training (WAJIB PAKAI HASIL SMOTE)
# Gunakan X_train_res dan y_train_encoded_res
lgbm_model_word2vec.fit(X_train_res_word2vec, y_train_encoded_word2vec)

# 3. Prediksi
y_pred_lgbm_word2vec = lgbm_model_word2vec.predict(X_test_w2v) # Changed to X_test_w2v

# 4. Evaluasi
print('LightGBM Accuracy:', accuracy_score(y_test_encoded_word2vec, y_pred_lgbm_word2vec))
print("\nLaporan Klasifikasi LightGBM:")
print(classification_report(y_test_encoded_word2vec, y_pred_lgbm_word2vec, target_names=le.classes_))